# **Joint _Fermi_-LAT and H.E.S.S. spectral analysis with GammaPy (and FermiPy) tutorial**

In this "tutorial" notebook, we will go over ...

## **_Fermi_-LAT setup and analysis in FermiPy**

The first step before the analysis is to obtain the data. This is achieved by downloading directly from the Fermi-LAT Data Server with the adequate parameters. For example:

* Object Name or Coordinates: 1ES0347-121
* Coordinate System: J2000
* Search Radius (Deg): 15
* Observation Dates: 2008-08-04 15:43:36, 2024-08-04 15:43:36 (YYYY-MM-DD)
* Time System: Gregorian
* Energy Range [MeV]: 100, 3000000
* LAT Data Type: Photon
* Spacecraft Data: Yes

We can obtain all of these files by running the bash script `get_files.sh`. This will give us a set of `PH.fits` files corresponding to the actual photon data (LT1), and `SC.fits` for spacecraft (LT2) data. The last command will also generate a **file list** in `.txt` format that links to each observation.

The next step is to prepare the `fermi_config.yaml` file which specifies exactly the selection and modeling to be carried out on the data. In our case, the configuration file may look like:
```
# INPUT/OUTPUT FILES
fileio:
  # Output directory and log file
  outdir     : "./fermi-out/"
  logfile    : "./fermi-out/fermi.log"
  # Use temporary work folder
  usescratch : False
  # scratchdir : '/work'

# DATA FILES
data:
  # Photon file or list of events (with absolute paths to fits files)
  evfile : "$FLAT_DATA/1es0347-121/events_list.txt"
  # Spacecraft file
  scfile : "$FLAT_DATA/1es0347-121/spacecraft.fits"
  # Livetime cube (if null, generate it)
  # ltcube : null

# BINNING: Control the spatial and spectral binning of the data.
binning:
  # Width of ROI in degrees
  roiwidth   : 15.0
  # Spatial bin size in degrees
  binsz      : 0.1
  # Number of energy bins per decade
  binsperdec : 2

# SELECTION
selection :
  # Energy (minimum) [MeV]
  emin    : 100
  # Energy (maximum) [MeV] - Limit to 300 GeV
  emax    : 300000
  # Maximum zenith angle (reduce contamination from Earth's background)
  zmax    : 90
  # Event class
  evclass : 128
  # Event type
  evtype  : 3
  # Time (start)
  tmin    : 239557417
  # Time (end)
  tmax    : 744479021
  # Right ascension of target
  ra      : 57.3466
  # Declination of target
  dec     : -11.9909

  # gtmktime parameters
  # select times when data quality is good (>0), lat in normal configuration (==1) 
  filter : 'DATA_QUAL>0 && LAT_CONFIG==1'
  # exclude time intervals when the Earth's limb intersects the selected region of interest
  # (current recommendation: no)
  roicut : 'no'

# GTLIKE: Control the setup of the likelihood analysis
gtlike:
  # Enable the correction for energy dispersion
  edisp : True
  # Number of bins to use for energy correction
  edisp_bins : 0 # DO NOT CHANGE edisp_bins will be handled by Gammapy
  # List of sources for which energy dispersion correction disabled
  edisp_disable : ['isodiff','galdiff']
  # Set the instrument response function
  irfs  : 'P8R3_SOURCE_V3'

# MODEL: Control the inclusion of point-source and diffuse components in the model
model:
  # Width of square region in degrees centered on the ROI that selects
  # sources for inclusion in the model
  src_roiwidth      : 20.0
  # Directory that will be searched for extended source FITS templates
  # (takes precedence over catalog source templates)
  extdir            : "$FERMIPY_DATA"
  # Path to IEM galactic diffuse mapcube
  galdiff           : 'gll_iem_v07.fits'
  # Path to isotropic diffuse template
  isodiff           : '$FERMIPY_DATA/iso_P8R3_SOURCE_V3_v1.txt'
  # Name of catalog to use
  catalogs          : ['gll_psc_v35.fit'] # ['4FGL']
```

(For more details, check the comments on the mentioned files or https://fermipy.readthedocs.io/en/latest/notebooks/SMC.html, https://fermipy.readthedocs.io/en/latest/notebooks/pg1553.html)

In order to prepare the data for compatibility with GammaPy, we must run a sequence of setup steps that filter the raw data and generate the files for IRFs, count maps, etc.

This setup step (and initial analysis of Fermi-LAT data) can be performed via FermiPy.

In [ ]:
from    fermipy.gtanalysis  import GTAnalysis
from    fermipy.plotting    import ROIPlotter

import  numpy               as np
import  matplotlib.pyplot   as plt

We can now instantiate the GTAnalysis class set `verbosity : 3` (Info mode) to supress Debug information. Calling `setup()` then generates all the required files and sets up everything for the likelihood analysis. Note: this usually takes a long time, especially for long time periods selected (14 years of data requires ~ 2 hours).

In [ ]:
gta = GTAnalysis("./fermi_config.yaml", logging = {'verbosity' : 3})
gta.setup()

Next, we run the commands that generate the PSF kernel and Edisp matrix for use in GammaPy. As per the recommendations of the GammaPy tutorial, we set `edisp_bins = 0` as the energy dispersion will be fully handled within GammaPy.

In [ ]:
gta.compute_psf(overwrite=True) # this creates the psf kernel
gta.compute_drm(edisp_bins=0, overwrite=True) # this creates the energy dispersion matrix

This produces a set of useful files:

- `ft1_00.fits`: Event list. This is generated by running gtselect and gtmktime on our input file list.

- `bexpmap_00.fits`: All-sky binned exposure map. This map is interpolated to create an exposure model when generating the srcmap file.

- `bexpmap_roi_00.fits`: Binned exposure map for the ROI. This file is only provided for visualization purposes in order to have an exposure map with the same binning as the data and model maps.

- `ccube_00.fits`: Counts cube for the ROI.

- `ltcube_00.fits`: Livetime cube. This contains a map of the livetime for this observation over the whole sky as a function of incidence angle.

- `srcmap_00.fits`: Source map cube. This file contains maps for each of the components in the ROI after convolution with exposure and the PSF. Note that energy dispersion is applied at run-time.

As well as some `.par` files that contain the parameters used in each called module. 

The setup is now completed, and in principle, the rest of the analysis can be carried out in GammaPy. However, it is useful to also perform a spectral analysis in FermiPy. This allows us to compare the results between both workflows, ensuring no problems appear when generating the files, and ensuring consistent results.

...

## **_Fermi_-LAT analysis with GammaPy**

## **H.E.S.S. analysis with GammaPy**

## **Joint _Fermi_-LAT and H.E.S.S. analysis with GammaPy**

Focus on just the joint _Fermi_-LAT and H.E.S.S. analysis, including bias. If interested, the user can run individual analyses as well. Assume time variability already done